In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Build a Text Cleaning Pipeline

In [24]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download if first time
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Cleans text data:
    - lowercase
    - remove URLs
    - remove emojis
    - remove symbols
    - remove stopwords
    - lemmatize or stem
    """

    # Convert to lowercase
    data = dataset.lower()

    # Remove URLs
    data = re.sub(r"http\S+|www\S+|https\S+", '', data)

    # Remove emojis
    data = re.sub(r'[^\x00-\x7F]+', '', data)

    # Remove unwanted characters
    data = re.sub(r'[^a-zA-Z\s]', '', data)

    # Tokenize
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize or Stem
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]

    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Text Classification using Machine Learning Models

### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


Importing libraries

In [25]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Download NLTK resources (run once)
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Loading dataset

In [26]:
import os
print(os.listdir())


['.config', 'drive', 'sample_data']


In [27]:
print(os.listdir('/content/drive/MyDrive/AI and ML week 8 workshop8'))


['AI and ML week8 Text Classification.ipynb', 'trumptweets_small.csv', '2025_W08_Text_Pre_processing_Questions.ipynb', '2025_W08_Text_Classification_Question.ipynb', 'trum_tweet_sentiment_analysis.csv', 'AI and ML week8 Text Pre-processing.ipynb']


Loading the dataset

In [28]:
df = pd.read_csv("/content/drive/MyDrive/AI and ML week 8 workshop8/trum_tweet_sentiment_analysis.csv")

In [29]:
# Show first 5 rows
print(df.head())

# Check required columns
print("\nColumns:", df.columns)



                                                text  Sentiment
0  RT @JohnLeguizamo: #trump not draining swamp b...          0
1  ICYMI: Hackers Rig FM Radio Stations To Play A...          0
2  Trump protests: LGBTQ rally in New York https:...          1
3  "Hi I'm Piers Morgan. David Beckham is awful b...          0
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...          0

Columns: Index(['text', 'Sentiment'], dtype='object')


In [30]:
df = df[['text', 'Sentiment']]
df.rename(columns={'Sentiment':'label'}, inplace=True)


In [31]:
# Keep only needed columns
df = df[['text', 'label']]

# Remove missing values
df.dropna(inplace=True)

print("\nDataset Shape:", df.shape)


Dataset Shape: (1850123, 2)


Text cleaning function

In [32]:

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Cleans text data:
    - lowercase
    - remove URLs
    - remove mentions
    - remove punctuation
    - remove special chars
    - remove stopwords
    - tokenize
    - lemmatize or stem
    """

    # Convert to lowercase
    data = dataset.lower()

    # Remove URLs
    data = re.sub(r"http\S+|www\S+|https\S+", '', data)

    # Remove mentions (@username)
    data = re.sub(r'@\w+', '', data)

    # Remove hashtags symbol only (#trump -> trump)
    data = re.sub(r'#', '', data)

    # Remove punctuation / special characters / numbers
    data = re.sub(r'[^a-zA-Z\s]', '', data)

    # Tokenization
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization or Stemming
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]

    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

Apply cleaning

In [33]:
df['clean_text'] = df['text'].apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))

print("\nCleaned Text Sample:")
print(df[['text', 'clean_text']].head())


Cleaned Text Sample:
                                                text  \
0  RT @JohnLeguizamo: #trump not draining swamp b...   
1  ICYMI: Hackers Rig FM Radio Stations To Play A...   
2  Trump protests: LGBTQ rally in New York https:...   
3  "Hi I'm Piers Morgan. David Beckham is awful b...   
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...   

                                          clean_text  
0  rt trump draining swamp taxpayer dollar trip a...  
1  icymi hacker rig fm radio station play antitru...  
2    trump protest lgbtq rally new york bbcworld via  
3  hi im pier morgan david beckham awful donald t...  
4  rt tech firm suing buzzfeed publishing unverif...  


Defining features and labels

In [34]:
X = df['clean_text']
y = df['label']

Training and testing split

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Size:", X_train.shape[0])
print("Testing Size:", X_test.shape[0])


Training Size: 1480098
Testing Size: 370025


TF-IDF Vectorization

In [36]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF Shape:", X_train_tfidf.shape)


TF-IDF Shape: (1480098, 5000)


Train Logistic Regression Model

In [37]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

Prediction

In [38]:
y_pred = model.predict(X_test_tfidf)

Evaluation

In [39]:
print("\nAccuracy Score:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy Score: 0.9261752584284846

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248842
           1       0.90      0.87      0.88    121183

    accuracy                           0.93    370025
   macro avg       0.92      0.91      0.92    370025
weighted avg       0.93      0.93      0.93    370025

